# Comparação — limiares vs Isolation Forest vs Autoencoder (Épico 9 / E9.4)

> O Limen é um protótipo acadêmico e **não** é um dispositivo médico.

| Enfoque | Onde roda | Artefato |
| ------- | --------- | -------- |
| **Limiares** (`thresholds`) | API / worker (default CI) | Regras HR/SpO2 |
| **Isolation Forest** | API / worker (`isolation_forest` / `hybrid`) | `models/vitals/isolation_forest.joblib` |
| **Autoencoder PyTorch** | **Só evidência** (notebook) | `train_vitals_autoencoder.ipynb` — **fora** do runtime |

Holdout = fixtures `vitals_{normal,medium,high}.csv` (labels `normal`/`anomaly`).
**Sem download** de Kaggle/PhysioNet neste notebook.

Ambiente: deps do `backend/` (sklearn) bastam para limiares + IF.
AE usa Torch se `requirements-ml.txt` estiver instalado; senão a linha AE
fica como *skipped* (ainda documentada na tabela).

In [ ]:
from __future__ import annotations

import csv
import sys
from pathlib import Path

import numpy as np

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "backend"))

from app.cases.vitals_engine import (  # noqa: E402
    FEATURE_COLUMNS,
    VitalsAnomalyEngine,
    resolve_if_model_path,
)

VITALS = ROOT / "data" / "fixtures" / "vitals"
SEED = 20260721
np.random.seed(SEED)


def load_fixture_rows() -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    for path in sorted(VITALS.glob("vitals_*.csv")):
        with path.open(newline="", encoding="utf-8") as fh:
            for row in csv.DictReader(fh):
                row["_file"] = path.name
                rows.append(row)
    return rows


rows = load_fixture_rows()
y_true = np.array([1 if r["label"] == "anomaly" else 0 for r in rows])
print(f"n={len(rows)} anomalies={int(y_true.sum())} files={sorted({r['_file'] for r in rows})}")

In [ ]:
def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    agreement = float((y_pred == y_true).mean())
    return {
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "agreement": round(agreement, 3),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


def pred_thresholds_row(row: dict[str, str]) -> int:
    hr = float(row["heart_rate"])
    spo2 = float(row["spo2"])
    return 1 if (hr >= 100 or spo2 < 95) else 0


y_thr = np.array([pred_thresholds_row(r) for r in rows])
m_thr = binary_metrics(y_true, y_thr)
print("thresholds (por ponto):", m_thr)

In [ ]:
import joblib

model_path = resolve_if_model_path()
payload = joblib.load(model_path)
if_model = payload["model"] if isinstance(payload, dict) else payload
X = np.asarray([[float(r[c]) for c in FEATURE_COLUMNS] for r in rows], dtype=float)
y_if = (if_model.predict(X) == -1).astype(int)
m_if = binary_metrics(y_true, y_if)
print("isolation_forest (por ponto):", m_if)
print("model:", model_path)

In [ ]:
# Autoencoder — só evidência (opcional Torch). Não roda na API/worker.
m_ae: dict[str, float | str]
try:
    import torch
    import torch.nn as nn

    class TinyAE(nn.Module):
        def __init__(self, dim: int = 5) -> None:
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(dim, 8), nn.ReLU(), nn.Linear(8, 3), nn.ReLU(),
                nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, dim),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)

    normals = X[y_true == 0]
    mu, sigma = normals.mean(axis=0), normals.std(axis=0)
    sigma = np.where(sigma < 1e-6, 1.0, sigma)
    Xn = (X - mu) / sigma
    train = torch.from_numpy(Xn[y_true == 0].astype(np.float32))
    ae = TinyAE(dim=X.shape[1])
    opt = torch.optim.Adam(ae.parameters(), lr=1e-2)
    loss_fn = nn.MSELoss()
    ae.train()
    for _epoch in range(25):
        opt.zero_grad()
        pred = ae(train)
        loss = loss_fn(pred, train)
        loss.backward()
        opt.step()
    ae.eval()
    with torch.no_grad():
        recon = ae(torch.from_numpy(Xn.astype(np.float32))).numpy()
    err = ((recon - Xn) ** 2).mean(axis=1)
    # Limiar = p95 do erro nos normais (holdout simples sobre fixtures).
    thr = float(np.quantile(err[y_true == 0], 0.95))
    y_ae = (err > thr).astype(int)
    m_ae = binary_metrics(y_true, y_ae)
    m_ae["note"] = "autoencoder (evidência; fora do runtime)"
    print("autoencoder (por ponto):", m_ae, "err_thr", round(thr, 4))
except ImportError:
    m_ae = {
        "precision": float("nan"),
        "recall": float("nan"),
        "agreement": float("nan"),
        "note": "skipped — instale notebooks/requirements-ml.txt",
    }
    print("AE skipped (sem torch). Runtime da API permanece sem AE.")

In [ ]:
rows_cmp = [
    {"enfoque": "thresholds (runtime CI)", "onde": "API/worker", **{k: m_thr[k] for k in ("precision", "recall", "agreement")}},
    {"enfoque": "isolation_forest (runtime opt-in)", "onde": "API/worker", **{k: m_if[k] for k in ("precision", "recall", "agreement")}},
    {
        "enfoque": "autoencoder (evidência)",
        "onde": "notebook only",
        "precision": m_ae.get("precision"),
        "recall": m_ae.get("recall"),
        "agreement": m_ae.get("agreement"),
    },
]
print(f"{'enfoque':40} {'onde':14} precision recall agreement")
for row in rows_cmp:
    print(
        f"{row['enfoque']:40} {row['onde']:14} "
        f"{row['precision']!s:>9} {row['recall']!s:>6} {row['agreement']!s:>9}"
    )
print("\nSérie-level (Caso): use VitalsAnomalyEngine(backend=...).")
for name in ("vitals_normal.csv", "vitals_medium.csv", "vitals_high.csv"):
    content = (VITALS / name).read_bytes()
    for backend in ("thresholds", "isolation_forest", "hybrid"):
        risk = VitalsAnomalyEngine(backend=backend).analyze_csv(content)
        print(f"{name:20} {backend:18} → {risk.level:5} score={risk.score:.2f}")

## Leitura para o relatório / demo

- **API/runtime:** `LIMEN_VITALS_BACKEND=thresholds|isolation_forest|hybrid`
  ([ADR 0029](../docs/adr/0029-vitais-ml-hibrido.md)). CI permanece em `thresholds`.
- **Demo:** `hybrid` (Compose / `.env`) — limiares **OU** IF.
- **AE:** evidência de epochs/loss em `train_vitals_autoencoder.ipynb` — **não**
  é valor de `LIMEN_VITALS_BACKEND`.
- Métricas acima são sobre fixtures sintéticas (agreement / precision / recall);
  não constituem validação clínica.